# Task 5 - Adversarial Image-Text Pair Generation & Robustness Testing

This notebook runs an end-to-end robustness benchmark:
- creates 100 adversarial image-text pairs from clean entailment examples
- runs text attacks (negation, color/object swap, synonym substitution, paraphrase)
- runs image attacks (FGSM + PGD at eps = 1/255, 4/255, 8/255)
- reports clean vs adversarial accuracy drop and attack success rates
- identifies vulnerable image regions and text tokens via gradient signals

All outputs are saved in `task5_artifacts/`.


In [ ]:
# Optional: install if your environment is missing dependencies.
# %pip install torch torchvision transformers pandas pillow nltk


In [ ]:
from task5_adversarial import Task5Config, run_task5

config = Task5Config(
    test_csv='cleaned_snli_ve_test.csv',
    checkpoint_path='final_sota_visual_entailment3.pth',
    output_dir='task5_artifacts',
    num_adversarial_pairs=100,
    epsilons=[1/255, 4/255, 8/255],
    pgd_steps=10,
    batch_size=8,
    max_text_length=64,
    clean_eval_limit=None,   # set to an int (e.g. 3000) for faster iteration
    save_adversarial_images=False,
)
config


In [ ]:
summary = run_task5(config)
summary


In [ ]:
import json
import pandas as pd
from pathlib import Path

out_dir = Path(config.output_dir)

with (out_dir / 'task5_robustness_summary.json').open() as f:
    summary = json.load(f)

print('Clean test accuracy      :', round(summary['clean_test_accuracy'], 4))
print('Text adversarial accuracy:', round(summary['text_adversarial_accuracy'], 4))
print('Combined adversarial acc :', None if summary['combined_adversarial_accuracy'] is None else round(summary['combined_adversarial_accuracy'], 4))

image_df = pd.read_csv(out_dir / 'task5_image_attack_results.csv')
print('
Image attack hardness table:')
image_df


In [ ]:
import pandas as pd
from pathlib import Path

out_dir = Path(config.output_dir)

token_vuln = pd.read_csv(out_dir / 'task5_token_vulnerability.csv')
region_vuln = pd.read_csv(out_dir / 'task5_image_region_vulnerability.csv')
text_pairs = pd.read_csv(out_dir / 'task5_text_adversarial_pairs.csv')

print('Top vulnerable tokens:')
display(token_vuln.head(15))

print('
Top vulnerable image patches (14x14 grid coordinates):')
display(region_vuln.head(10))

print('
Sample adversarial text pairs:')
display(text_pairs.head(10))
